        # Lesson 7.8 · Practical Assignment: The Forecasting Experiment
        ### Your end-to-end Phase 7 project: from raw prices to an honest model comparison

        | | |
        |---|---|
        | ⏱️ **Time** | about 180 minutes |
        | 📚 **Before you start** | Lessons 7.1 – 7.7 |
        | 🧪 **Format** | lesson → worked examples → exercises with hints → recap |

        ## 🎯 What you will be able to do after this notebook
        - Carry out a complete, **leak-free** forecasting study on real-world-style messy market data
- Visualise, decompose, baseline, model (ARIMA-family, linear ML, Random Forest, XGBoost) and **compare by horizon**
- Write a short, decision-oriented **report** a cooperative manager can act on

> 🌾 **The story so far.** The Highland market is the cooperative's hardest case: a washed-out road left gaps, and a clerk's decimal slips left outliers. The board wants to know **how far ahead prices can be forecast, and which method to trust**. This is your engagement as the cooperative's analyst.

In [1]:
import sys, pathlib, warnings
# Find the course folder (works no matter where Jupyter was started)
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "agrilab.py").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import agrilab as ag          # our helper library: src/agrilab.py

from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
warnings.filterwarnings("ignore")   # (after imports: statsmodels re-enables its own warnings when imported)
ag.setup_plots()
ag.set_seed(42)
print("Setup complete. Course folder:", ROOT.name)

Setup complete. Course folder: course


## The brief

> **To:** Analyst  **From:** Cooperative board
> We sell tomatoes through the **Highland** market. We must decide, each month, whether to sell now or store for up to **8 weeks**.
> Please build and *fairly* evaluate forecasting models for **weekly** Highland tomato prices, and tell us **which model to use and how far to trust it**.

### Rules of the game
1. **No leakage.** Every model sees only data available at its forecast origin. (We will test for it.)
2. **Walk-forward evaluation** for every model: first window 300 weeks, forecast horizon 8 weeks, a new origin every 8 weeks.
3. **Baselines first.** A model that can't beat naive and seasonal-naive is not a model.
4. Report **error by horizon** and a plain-language conclusion.

### How you'll be graded (self-assessment rubric)
| Criterion | Points | What full marks looks like |
|---|---|---|
| Data cleaning | 15 | Outliers removed, gaps repaired, regular weekly index, **no future information** |
| Exploration | 15 | Clear, labelled history plot; trend growth, seasonality strength, peak/trough months |
| Baselines | 10 | Naive, seasonal naive and moving average evaluated by walk-forward |
| Statistical model | 15 | ARIMA-family model with seasonality (Fourier) and a rain driver, correctly back-transformed |
| ML models | 20 | Ridge, Random Forest, XGBoost using the direct strategy and leak-free features |
| Comparison & insight | 15 | Leaderboard, error-by-horizon plot, skill vs naive, win-rate |
| Report | 10 | Concise, honest, decision-oriented, notes limitations |

✅ Each step has an auto-check. Get them all green, *then* write your report.

---
## Step 1 · Load and clean the data  *(15 pts)*
Pipeline: load raw Highland tomato records → flag **decimal slips** (price > 3× or < ⅓ of the 31-observation centered rolling median) →
set them to missing → put on a **daily** calendar and interpolate by time (this repairs random gaps *and* the washed-out road) → average to **weekly** (week ending Sunday).

In [2]:
COMMODITY, MARKET = "tomato", "Highland"        # keep these until every check is green; then explore other series
EVAL = dict(initial=300, horizon=8, step=8)     # the evaluation protocol everyone must use

---
### 🧩 Exercise 1: A cleaning function  ⭐⭐

Write `clean_weekly(commodity, market)` that returns a **weekly** `pd.Series` (DatetimeIndex, week ending Sunday) as described above.
Call it to create `y`.

<details><summary>💡 Hint 1: gentle nudge</summary>

`raw = ag.load_prices(commodity, market).set_index('date')['price_per_kg']`

</details>

<details><summary>💡 Hint 2: bigger nudge</summary>

`med = raw.rolling(31, center=True, min_periods=5).median()`; `bad = (raw > 3*med) | (raw < med/3)`; `clean = raw.mask(bad)`.

</details>

<details><summary>💡 Hint 3: almost the answer</summary>

`daily = clean.resample('D').mean().interpolate('time')`; return `daily.resample('W').mean()`.

</details>

*Write your answer in the next cell, then run the check cell below it. Stuck after the hints? The full solution is in `solutions/`.*

In [ ]:
def clean_weekly(commodity, market):
    # your code
    return None

y = clean_weekly(COMMODITY, MARKET)

In [ ]:
try:
    ag.check("a Series with a DatetimeIndex", isinstance(y, pd.Series) and isinstance(y.index, pd.DatetimeIndex))
    ag.check("523 regular weekly observations, no NaN", y is not None and len(y) == 523 and y.isna().sum() == 0)
    ag.check("weekly frequency", y.index.freqstr is not None and y.index.freqstr.startswith("W"))
    ag.check("decimal slips are gone (no price above 5x the median)", y.max() < 5 * y.median())
except Exception as _e:
    print('\u23f3 Not ready yet: finish the exercise cell above, then re-run this check.  (' + type(_e).__name__ + ': ' + str(_e)[:90] + ')')

---
## Step 2 · Visualise the history  *(part of 15 pts)*
A manager reads charts before tables. Produce **one** figure (store the `Figure` in `fig_hist`) showing the weekly price and its
52-week rolling mean. It must have a **title** and a **y-axis label**.

---
### 🧩 Exercise 2: The history chart  ⭐

Create `fig_hist, ax = plt.subplots(figsize=(11, 4))`, plot `y` and `y.rolling(52).mean()`, set a title and a y-label, add a legend.

<details><summary>💡 Hint 1: gentle nudge</summary>

`y.plot(ax=ax, label='weekly price')` and `y.rolling(52).mean().plot(ax=ax, lw=2.5, label='52-week mean')`.

</details>

<details><summary>💡 Hint 2: bigger nudge</summary>

`ax.set(title='...', ylabel='price per kg')`; `ax.legend()`.

</details>

*Write your answer in the next cell, then run the check cell below it. Stuck after the hints? The full solution is in `solutions/`.*

In [ ]:
fig_hist, ax = plt.subplots(figsize=(11, 4))
# plot, title, ylabel, legend
plt.show()

In [ ]:
try:
    ax0 = fig_hist.axes[0]
    ag.check("figure has a title", len(ax0.get_title()) > 3)
    ag.check("y axis is labelled", len(ax0.get_ylabel()) > 2)
    ag.check("at least two lines plotted", len(ax0.lines) >= 2)
except Exception as _e:
    print('\u23f3 Not ready yet: finish the exercise cell above, then re-run this check.  (' + type(_e).__name__ + ': ' + str(_e)[:90] + ')')

---
## Step 3 · Trend and seasonality analysis  *(part of 15 pts)*
Fit `STL(np.log(y), period=52, robust=True)`. From the result compute:
- `fs`: seasonal strength: `max(0, 1 - Var(resid)/Var(seasonal + resid))`
- `growth_pct`: average **annual % growth** of the trend: `(exp((trend_end - trend_start)/years) - 1) × 100`, with `years = (last date − first date).days / 365.25`
- `peak_month`, `trough_month`: the calendar months (1–12) whose **mean seasonal component** is highest / lowest.

---
### 🧩 Exercise 3: Decompose and summarise  ⭐⭐

Compute `fs`, `growth_pct`, `peak_month`, `trough_month` as defined above.

<details><summary>💡 Hint 1: gentle nudge</summary>

`res = STL(np.log(y), period=52, robust=True).fit()`; then `res.trend`, `res.seasonal`, `res.resid` are Series.

</details>

<details><summary>💡 Hint 2: bigger nudge</summary>

Month means: `res.seasonal.groupby(res.seasonal.index.month).mean()` → use `.idxmax()` / `.idxmin()`.

</details>

<details><summary>💡 Hint 3: almost the answer</summary>

Growth: `years = (y.index[-1] - y.index[0]).days / 365.25`; `(np.exp((res.trend.iloc[-1] - res.trend.iloc[0]) / years) - 1) * 100`.

</details>

*Write your answer in the next cell, then run the check cell below it. Stuck after the hints? The full solution is in `solutions/`.*

In [ ]:
fs = growth_pct = peak_month = trough_month = None

In [ ]:
try:
    ag.check("seasonal strength is high (0.7 - 0.95)", fs is not None and 0.7 < fs < 0.95)
    ag.check("annual trend growth is between 4 % and 20 %", growth_pct is not None and 4 < growth_pct < 20)
    ag.check("peak price falls in spring (month 3 or 4)", peak_month in (3, 4))
    ag.check("trough falls in mid-year (months 6-8)", trough_month in (6, 7, 8))
except Exception as _e:
    print('\u23f3 Not ready yet: finish the exercise cell above, then re-run this check.  (' + type(_e).__name__ + ': ' + str(_e)[:90] + ')')

---
## Step 4 · Baselines and the moving average  *(10 pts)*
Every forecaster is a function `(train: Series, h: int) -> array of h forecasts`.
Build the dictionary `res` (model name → walk-forward DataFrame) containing **three** entries:
`"naive"`, `"seasonal naive"` and `"MA(4)"`: the last is the mean of the final four weeks repeated *h* times.
Use `ag.walk_forward(y, forecaster, **EVAL)`.

---
### 🧩 Exercise 4: Three baselines by walk-forward  ⭐

Fill `res` with the three baselines.

<details><summary>💡 Hint 1: gentle nudge</summary>

`ag.naive_forecast` and `ag.seasonal_naive_forecast` are ready-made forecasters.

</details>

<details><summary>💡 Hint 2: bigger nudge</summary>

For MA(4): `lambda train, h: np.repeat(train.iloc[-4:].mean(), h)`.

</details>

<details><summary>💡 Hint 3: almost the answer</summary>

`res['naive'] = ag.walk_forward(y, ag.naive_forecast, **EVAL)`: `**EVAL` unpacks the dict into keyword arguments.

</details>

*Write your answer in the next cell, then run the check cell below it. Stuck after the hints? The full solution is in `solutions/`.*

In [ ]:
res = {}
# res["naive"] = ...

In [ ]:
try:
    ag.check("three baselines present", {"naive", "seasonal naive", "MA(4)"}.issubset(res))
    ag.check("each covers horizons 1..8", all(set(r.h) == set(range(1, 9)) for r in res.values()))
    ag.check("naive MAE matches the reference protocol", np.isclose(ag.mae(res["naive"].actual, res["naive"].forecast),
             ag.mae(*[(lambda r: (r.actual, r.forecast))(ag.walk_forward(y, ag.naive_forecast, 300, 8, 8))][0])))
except Exception as _e:
    print('\u23f3 Not ready yet: finish the exercise cell above, then re-run this check.  (' + type(_e).__name__ + ': ' + str(_e)[:90] + ')')

---
## Step 5 · ARIMA / SARIMA family  *(15 pts)*
Weekly data with a 52-week season is expensive for SARIMA, so use the pattern from Lesson 7.5: **SARIMAX(2,1,1) on log price** with
**Fourier terms** (yearly seasonality) and the **8-week-lagged rainfall** driver. The exogenous table `EXOG_FULL` is provided below.
The model works in logs → exponentiate the forecasts.

In [11]:
def fourier(index, K=3, period=365.25):
    doy = index.dayofyear.to_numpy()
    cols = {}
    for k in range(1, K + 1):
        cols[f"sin{k}"] = np.sin(2 * np.pi * k * doy / period)
        cols[f"cos{k}"] = np.cos(2 * np.pi * k * doy / period)
    return pd.DataFrame(cols, index=index)

rain = ag.load_weather().set_index("date")["rainfall_mm"].resample("W").sum().reindex(y.index)
rain9 = rain.rolling(9, min_periods=1).sum()
rain_lag8 = rain9.shift(8).bfill()
EXOG_FULL = pd.concat([fourier(y.index), ((rain_lag8 - rain_lag8.mean()) / rain_lag8.std()).rename("rain_lag8")], axis=1)
EXOG_DF = pd.DataFrame({"rain_9w": rain9})      # for the ML models in Steps 6-8
EXOG_FULL.columns.tolist()

['sin1', 'cos1', 'sin2', 'cos2', 'sin3', 'cos3', 'rain_lag8']

---
### 🧩 Exercise 5: SARIMAX forecaster  ⭐⭐⭐

Write `sarimax_fp(train, h)`:
1. Locate the position of the last training week in `EXOG_FULL.index`.
2. Fit `SARIMAX(np.log(train), exog=EXOG_FULL.loc[train.index], order=(2, 1, 1)).fit(disp=False)`.
3. Forecast `h` steps with `exog` = the **next `h` rows** of `EXOG_FULL` (Fourier terms are known; lagged rain is already observed).
4. Return `np.exp(forecast).to_numpy()`.
Then evaluate with walk-forward and store in `res["SARIMAX"]`.

<details><summary>💡 Hint 1: gentle nudge</summary>

`i = EXOG_FULL.index.get_loc(train.index[-1])`; future exog: `EXOG_FULL.iloc[i+1 : i+1+h]`.

</details>

<details><summary>💡 Hint 2: bigger nudge</summary>

`fit = SARIMAX(np.log(train), exog=EXOG_FULL.loc[train.index], order=(2,1,1)).fit(disp=False)`

</details>

<details><summary>💡 Hint 3: almost the answer</summary>

`return np.exp(fit.forecast(h, exog=future)).to_numpy()`

</details>

*Write your answer in the next cell, then run the check cell below it. Stuck after the hints? The full solution is in `solutions/`.*

In [ ]:
def sarimax_fp(train, h):
    # your code
    return None

# res["SARIMAX"] = ag.walk_forward(y, sarimax_fp, **EVAL)

In [ ]:
try:
    f = sarimax_fp(y.iloc[:300], 8)
    ag.check("returns 8 forecasts in price units", f is not None and len(f) == 8 and 20 < np.mean(f) < 300, "np.exp()!")
    ag.check("SARIMAX beats naive overall", "SARIMAX" in res and ag.mae(res["SARIMAX"].actual, res["SARIMAX"].forecast) < ag.mae(res["naive"].actual, res["naive"].forecast))
except Exception as _e:
    print('\u23f3 Not ready yet: finish the exercise cell above, then re-run this check.  (' + type(_e).__name__ + ': ' + str(_e)[:90] + ')')

---
## Steps 6–8 · Machine-learning models  *(20 pts)*
Use the **direct** strategy via `ag.direct_ml_forecaster(make_model, exog=EXOG_DF)`, where `make_model` is a zero-argument function returning a fresh model.
Build three entries in `res`:

| Key | Model |
|---|---|
| `"Ridge (lags)"` | `StandardScaler` + `Ridge(alpha=10)` pipeline |
| `"Random Forest"` | `RandomForestRegressor(100, min_samples_leaf=3, random_state=0, n_jobs=-1)` |
| `"XGBoost"` | `XGBRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=0, n_jobs=1)` |

*(Running all three takes about a minute; a good moment for tea.)*

---
### 🧩 Exercise 6: Three ML forecasters  ⭐⭐

Create the three entries in `res` (walk-forward with `**EVAL`).

<details><summary>💡 Hint 1: gentle nudge</summary>

`ag.direct_ml_forecaster(lambda: make_pipeline(StandardScaler(), Ridge(10)), EXOG_DF)` returns a forecaster.

</details>

<details><summary>💡 Hint 2: bigger nudge</summary>

Each model gets its own `lambda: Model(...)`. Don't instantiate the model *outside* the lambda: you need a fresh one for every refit.

</details>

<details><summary>💡 Hint 3: almost the answer</summary>

`res['Ridge (lags)'] = ag.walk_forward(y, fp_ridge, **EVAL)` and likewise for the others.

</details>

*Write your answer in the next cell, then run the check cell below it. Stuck after the hints? The full solution is in `solutions/`.*

In [ ]:
# fp_ridge = ag.direct_ml_forecaster(...)
# res["Ridge (lags)"] = ...

In [ ]:
try:
    need = {"Ridge (lags)", "Random Forest", "XGBoost"}
    ag.check("all three ML models evaluated", need.issubset(res))
    naive_mae = ag.mae(res["naive"].actual, res["naive"].forecast)
    ag.check("every ML model beats naive", all(ag.mae(res[k].actual, res[k].forecast) < naive_mae for k in need if k in res))
    ag.check("same origins as the baselines (fair comparison)", all(res[k].origin_date.nunique() == res["naive"].origin_date.nunique() for k in need if k in res))
except Exception as _e:
    print('\u23f3 Not ready yet: finish the exercise cell above, then re-run this check.  (' + type(_e).__name__ + ': ' + str(_e)[:90] + ')')

---
## Step 9 · Compare the models  *(15 pts)*
Produce:
- `board = ag.score_table(res, y_train=y.iloc[:300])` (sorted by MAE)
- `best_model`: the name at the top of the board
- `skill_vs_naive`: `1 - MAE_best / MAE_naive` (e.g. 0.45 = 45 % lower error than naive)
- `win_rate`: the share of (origin, horizon) forecasts where the best model's absolute error is **smaller than naive's**
- an error-by-horizon DataFrame `by_h` (rows = horizon, columns = model) and a line plot of it

---
### 🧩 Exercise 7: Leaderboard, skill and win-rate  ⭐⭐

Compute `board`, `best_model`, `skill_vs_naive`, `win_rate`, `by_h`, and plot `by_h`.

<details><summary>💡 Hint 1: gentle nudge</summary>

`by_h = pd.DataFrame({n: r.groupby('h').apply(lambda d: ag.mae(d.actual, d.forecast)) for n, r in res.items()})`

</details>

<details><summary>💡 Hint 2: bigger nudge</summary>

Win-rate: both DataFrames list forecasts in the same order, so `np.mean(np.abs(b.actual - b.forecast).to_numpy() < np.abs(n.actual - n.forecast).to_numpy())`, with `b = res[best_model]`, `n = res['naive']`.

</details>

<details><summary>💡 Hint 3: almost the answer</summary>

`skill_vs_naive = 1 - board.loc[best_model, 'MAE'] / board.loc['naive', 'MAE']`

</details>

*Write your answer in the next cell, then run the check cell below it. Stuck after the hints? The full solution is in `solutions/`.*

In [ ]:
board = best_model = skill_vs_naive = win_rate = by_h = None

In [ ]:
try:
    ag.check("board has all 7 models", len(board) == 7)
    ag.check("best_model is top of the board and not a baseline", best_model == board.index[0] and best_model not in ("naive", "seasonal naive", "MA(4)"))
    ag.check("skill_vs_naive is a sensible fraction", 0.2 < skill_vs_naive < 0.9)
    ag.check("win_rate is a share between 0.5 and 1", 0.5 < win_rate <= 1)
    ag.check("by_h is 8 horizons x 7 models", by_h.shape == (8, 7))
except Exception as _e:
    print('\u23f3 Not ready yet: finish the exercise cell above, then re-run this check.  (' + type(_e).__name__ + ': ' + str(_e)[:90] + ')')

---
## Step 10 · Write the report  *(10 pts)*
Write **6–10 sentences** in the string `report` for the cooperative board. Cover:
1. which model you recommend, and its skill vs naive at **h = 1, 4 and 8** (use numbers),
2. what the data say about **seasonality** (when to sell/store),
3. one **limitation or risk** (e.g. synthetic data, few origins, shocks you can't forecast),
4. one **next step**.

> ✍️ Good reports are short, specific, honest. *"XGBoost is best (MAE 9.1 vs naive 18.2)"* beats *"the ML models performed well."*

---
### 🧩 Exercise 8: Board report  ⭐

Write your report into the string `report`.

<details><summary>💡 Hint 1: gentle nudge</summary>

Start with the recommendation: model name + a number.

</details>

<details><summary>💡 Hint 2: bigger nudge</summary>

Use `by_h.loc[[1, 4, 8]].round(1)` to quote horizon-specific errors.

</details>

<details><summary>💡 Hint 3: almost the answer</summary>

Mention peak month and trough month from Step 3; those drive the sell/store decision.

</details>

*Write your answer in the next cell, then run the check cell below it. Stuck after the hints? The full solution is in `solutions/`.*

In [ ]:
report = """
Recommendation: ...
Seasonality: ...
Limitation: ...
Next step: ...
"""

In [ ]:
try:
    ag.check("report is a string of reasonable length (300-2500 chars)", isinstance(report, str) and 300 < len(report) < 2500)
    ag.check("it names a model and mentions the naive baseline", any(m in report for m in ["Ridge", "Forest", "XGBoost", "SARIMAX"]) and "naive" in report.lower())
    ag.check("it mentions seasonality and a limitation", "season" in report.lower() and "limit" in report.lower())
    ag.check("it contains numbers", sum(ch.isdigit() for ch in report) >= 8)
except Exception as _e:
    print('\u23f3 Not ready yet: finish the exercise cell above, then re-run this check.  (' + type(_e).__name__ + ': ' + str(_e)[:90] + ')')

---
## 🚀 Stretch goals (optional but recommended)
1. Switch `COMMODITY`/`MARKET`: do the same models win for *maize* at the *Harbor* market?
2. **Pool** all 3 markets × 1 crop into one XGBoost model with a `market` feature: does sharing data help?
3. Add **Fourier calendar features** and a **fuel-index** driver to the ML table.
4. Tune the Random Forest and XGBoost with `TimeSeriesSplit`: is the gain real in walk-forward?
5. Produce **prediction intervals** (e.g. quantile regression with XGBoost `objective="reg:quantileerror"`) and check their *coverage*.
6. Run the evaluation with `step=2` instead of 8: how stable is the ranking?

## ✅ Submission checklist
- [ ] All checks green · [ ] Figures titled & labelled · [ ] `res` has 7 models · [ ] Report written · [ ] Notebook runs top-to-bottom after *Restart & Run All*
- [ ] Leak test: *what would change if you accidentally used `rolling(center=True)`?* (Write one sentence in your notes.)

        ---
        ## 🧾 Recap
        - You ran a complete forecasting study: **clean → explore → baseline → model → compare → communicate**.
- The winner is the model that beats naive **at the horizons the decision needs**, with an honest walk-forward protocol.
- Simple, well-featured models are often within a whisker of complex ones, and cheaper to maintain.

**Next up:** Phase 8: neural networks from first principles: neurons, back-propagation, PyTorch, CNNs, sequence models, Transformers, and finally an LSTM price forecaster that you'll benchmark against today's leaderboard.